In [3]:
from dotenv import load_dotenv
import os
import requests
import json


load_dotenv()


True

# Task 1: Text Generation Temperature
 

In [41]:
from langchain_community.chat_models import ChatOpenAI
from langchain_openai import ChatOpenAI

model_low = ChatOpenAI(
    model="nvidia/nemotron-3-nano-30b-a3b:free",
    temperature=0,
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY"),
)

model_high = ChatOpenAI(
    model="nvidia/nemotron-3-nano-30b-a3b:free",
    temperature=0.9,
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY"),
)

In [42]:
message_low = model_low.invoke("what is Ramadan? make it super short and be poetic in your response")
print(message_low.content)


Ramadan— themoon’s quiet hush,  
a month of fasting, prayer, and light,  
where hearts feast on mercy,  
and every sunrise writes a new verse of devotion.


In [ ]:
message_high = model_high.invoke("what is Ramadan? make it super short and be poetic in your response")
print(message_high.content)


Moonlight feeds thesoul in Ramadan.  
*Short, poetic, and true.* ✨


In [14]:
message_low = model_low.invoke("tell me about the desert using only emojis, Format it as JSON")
print(message_low.content)


{
  "🏜️":"🌵🌞🔥🏜️🐪🌙🌌"
}


In [15]:
message_high = model_high.invoke("tell me about the desert using only emojis, Format it as JSON")
print(message_high.content)

{
  "environment": "🏜️",
  "temperature": "🔥",
  "flora": "🌵",
  "fauna": "🐪",
  "water": "💧",
  "night": "🌙",
  "sky": "☀️",
  "rock": "🪨"
}


## Task 2: Sentiment Analysis


In [59]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser

class SentimentResult(BaseModel):
    sentiment: Literal["positive", "neutral", "negative"] = Field(
        description="Sentiment label"
    )

parser = PydanticOutputParser(pydantic_object=SentimentResult)

In [60]:
sentences = [
    "Kindness creates lasting joy.",
    "Success rewards persistent effort.",
    "I love Sunlight. It warms the skin.",
    "Pessemestic all the time.",
    "The storm caused damage!",
    "The clock ticks steadily."
]

for sentence in sentences:
    prompt = f"""
Classify the sentiment of this sentence:

"{sentence}"

Return ONLY valid JSON.
Do not use markdown.

{parser.get_format_instructions()}
"""

    response = model_low.invoke(prompt)
    result = parser.parse(response.content)

    print(sentence, " ::", result.sentiment)

Kindness creates lasting joy.  :: positive
Success rewards persistent effort.  :: positive
I love Sunlight. It warms the skin.  :: positive
Pessemestic all the time.  :: negative
The storm caused damage!  :: negative
The clock ticks steadily.  :: neutral


# Task 3: Categorization

In [61]:
from typing import List 
from pydantic import BaseModel, Field
class Tags(BaseModel):
    tags: List[str] = Field(..., description="List of relevant tags. Output ONLY from: 'cars', 'shopping', 'sports', 'study', 'work'.")

In [62]:
catt= model_low.with_structured_output(Tags)

In [63]:
from langchain_core.messages import SystemMessage, HumanMessage

sentences = [
    "That restoration looks incredible; you have a real talent for mechanics.",
    "I found the perfect gift today! The staff was incredibly helpful.",
    "Great game today! Your teamwork was the key to that victory.",
    "Learning together helped me finally grasp these concepts. Thank you!"
]
for sentence in sentences:
    messages = [
        SystemMessage("You are a Categorizor. You must respond with ONLY with categories givin."),
        HumanMessage(sentence)
    ]

result = catt.invoke(messages)
print(f"{sentence} → {result.tags}")


Learning together helped me finally grasp these concepts. Thank you! → ['Positive Feedback']


In [71]:
class CV(BaseModel):
    name: str = Field(None, description="Full name of the candidate.")
    email: str = Field(None, description="Email address of the candidate.")
    skills: str = Field(None, description="Skills of the candidate.")
    experience: str = Field(None, description="Work experience of the candidate.")
    education: str = Field(None, description="Education of the candidate.")

In [72]:
extr_cv= model_low.with_structured_output(CV)

In [73]:
import pypdf
cv_path = r"C:\Users\MA\Desktop\pxd\alex_johnson_cv.pdf"
reader = pypdf.PdfReader(cv_path)
cv_text = "\n".join([page.extract_text() for page in reader.pages])

messages = [
        SystemMessage("You are a cv data extractor extract info based on the CV given"),
        HumanMessage(sentence)
    ]
result = extr_cv.invoke(cv_text)

print("Name:", result.name)
print("Email:", result.email)
print("Skills:", result.skills)
print("Experience:", result.experience)
print("EDUCATION:", result.education)

Name: Alex Johnson
Email: alex.johnson@email.com
Skills: Python, SQL, JavaScript, scikit-learn, AutoGluon, SHAP, LangChain, HuggingFace, XGBoost, pandas, numpy, matplotlib, seaborn, Plotly, Gradio, FastAPI, Docker, Git, Jupyter
Experience: Machine Learning Engineer — DataCore Inc. (Jan 2022 – Present)
- Built classification pipelines using scikit-learn Pipeline + ColumnTransformer, reducing data leakage incidents to zero
- Deployed AutoGluon models achieving ROC-AUC > 0.92 on tabular datasets with 100K+ rows
- Used SHAP to explain model predictions to non-technical stakeholders, improving trust and adoption
- Reduced hotel booking cancellation rate by 15% through ML-driven retention offer system

Data Analyst — InsightHub (Jun 2020 – Dec 2021)
- Performed EDA on healthcare datasets using pandas, seaborn, and matplotlib
- Built regression models to predict patient readmission rates (RMSE reduced by 22%)
- Developed interactive dashboards for clinical teams using Plotly and Gradio
EDUCAT

In [74]:
def add(a, b):
    return a + b

def subtract(a, b):
    return a - b

def multiply(a, b):
    return a * b

def divide(a, b):
    if b == 0:
        raise ValueError('Cannot divide by zero')
    return a / b



In [76]:
system_prompt = """
You are a math assistant. You have access to these tools:

def add(a, b):
    return a + b

def subtract(a, b):
    return a - b

def multiply(a, b):
    return a * b

def divide(a, b):
    return a / b

Return the tool name and arguments required to solve the user's math request.
Only return the function name and its arguments — do not compute the result yourself.
"""



In [77]:
from pydantic import BaseModel, Field
from typing import Dict, Any

class ToolCall(BaseModel):
    tool_name: str          = Field(description="The name of the function to use")
    arguments: Dict[str, Any] = Field(description="The parameters to pass")

In [78]:
tool_math   = model_low.with_structured_output(ToolCall)

In [95]:
question = 'what is two plus 5'

messages = [
    SystemMessage(system_prompt),
    HumanMessage(question)
]

tool_call= tool_math.invoke(messages)
result = fn(**tool_call.arguments)
print(f"Answer: {result}")


RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '50', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1777161600000'}, 'provider_name': None}}, 'user_id': 'user_3Cl5BfcUtKiPXJnBXe4dNqCKQD2'}